# 🚀 SUSA AURA — Master Google Colab GPU Training & Evaluation Pipeline

> **Adaptive Universal Reasoning Assistant (SUSA AURA): An original decoder-only Transformer built and trained from scratch.**

---

### 📌 Zero-Cost & Originality Principles:
- **Hardware Cost:** ₹0 (leveraging Google Colab Free Tier GPUs — NVIDIA T4 / V100 / A100).
- **Architecture:** Original decoder-only Transformer with RoPE, SwiGLU, RMSNorm, Grouped-Query Attention (GQA), and tied embeddings.
- **Persistent Checkpoints:** Local high-speed NVMe storage continuously synced to Google Drive for disconnection recovery.
- **Subsystems:** AURA Model, AURA Core, AURA Connect, AURA Memory, and AURA Tools.

## Step 1: Environment & Hardware Diagnostics (Python, PyTorch, CUDA, GPU Memory)

In [ ]:
#@title Step 1: Verify Hardware & GPU Environment { display-mode: "form" }
import sys
import torch
import psutil

print("=" * 70)
print("  SUSA AURA — HARDWARE & ENVIRONMENT DIAGNOSTICS")
print("=" * 70)
print(f"  Python Version:     {sys.version.split()[0]}")
print(f"  PyTorch Version:    {torch.__version__}")
print(f"  System RAM Total:   {psutil.virtual_memory().total / (1024**3):.2f} GiB")
print(f"  System RAM Avail:   {psutil.virtual_memory().available / (1024**3):.2f} GiB")

cuda_avail = torch.cuda.is_available()
print(f"  CUDA GPU Available: {cuda_avail}")
if cuda_avail:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  Active GPU:         {gpu_name} ({gpu_vram:.2f} GiB VRAM)")
    print("=" * 70)
    print("[✓] Environment ready for accelerated training!")
else:
    print("=" * 70)
    print("[!] NOTICE: Running on CPU. For faster training, switch Runtime -> Change runtime type -> T4 GPU.")

## Step 2: Clone or Update SUSA AURA Repository

In [ ]:
#@title Step 2: Obtain Repository from GitHub { display-mode: "form" }
import os

WORKSPACE_DIR = "/content/susa-aura"
REPO_URL = "https://github.com/saketh-nandu/susa-1b-codemind.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

if os.path.exists(WORKSPACE_DIR):
    print(f"Directory {WORKSPACE_DIR} already exists. Pulling latest code...")
    %cd {WORKSPACE_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    print(f"Cloning {REPO_URL} into {WORKSPACE_DIR}...")
    !git clone --branch {BRANCH} {REPO_URL} {WORKSPACE_DIR}
    %cd {WORKSPACE_DIR}

!pwd
!git log -n 1 --oneline

## Step 3: Install Package & Verify Dependencies

In [ ]:
#@title Step 3: Install Dependencies { display-mode: "form" }
!pip install --upgrade pip
!pip install -e ".[colab,dev]"
!pip install psutil pyyaml tokenizers tqdm datasets


## Step 4: Mount Google Drive (Crash Recovery & Checkpoint Persistence)

In [ ]:
#@title Step 4: Mount Google Drive (Optional) { display-mode: "form" }
ENABLE_DRIVE_SYNC = True #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/SUSA_AURA_Checkpoints" #@param {type:"string"}

if ENABLE_DRIVE_SYNC:
    try:
        from google.colab import drive
        print("Authorizing Google Drive mount...")
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f"[✓] Persistent checkpoint backup ready at: {DRIVE_DIR}")
    except Exception as e:
        print(f"[!] Drive mount skipped or failed: {e}")
else:
    print("Drive synchronization disabled. Checkpoints will be kept in local runtime storage.")

## Step 5: Select Architecture Profile & Estimate Resource Feasibility

In [ ]:
#@title Step 5: Select Architecture Profile { display-mode: "form" }
PROFILE = "smoke" #@param ["smoke", "small", "medium", "target_1b"]

import yaml
from aura.model.config import ModelConfig
from aura.utils.resource_estimator import estimate_resources

cfg_path = f"configs/{PROFILE}.yaml"
with open(cfg_path, "r", encoding="utf-8") as f:
    raw_cfg = yaml.safe_load(f)

model_cfg = ModelConfig.from_dict(raw_cfg)
print(model_cfg.print_parameter_summary())

batch_size = raw_cfg.get("training", {}).get("batch_size", 1)
res = estimate_resources(model_cfg, batch_size=batch_size, seq_len=model_cfg.max_seq_len)
print(res.summary())

## Step 6: Smoke Test Architecture & Pytest Suite

In [ ]:
#@title Step 6: Run Architecture Smoke Test & Tests { display-mode: "form" }
!python scripts/smoke_test_model.py
!pytest tests/ -v

## Step 7: Train Tokenizer & Prepare Binary Shards

In [ ]:
#@title Step 7: Prepare Tokenizer and Dataset Shards { display-mode: "form" }
TOKENIZER_DIR = f"tokenizer_cache/{PROFILE}"
VOCAB_SIZE = model_cfg.vocab_size

# 1. Download or generate sample dataset
!python scripts/download_dataset.py --source stdlib --output-dir data/raw_code --generate-instructions

# 2. Train Byte-Level BPE Tokenizer
!python scripts/train_tokenizer.py --input-dir data/raw_code --vocab-size {VOCAB_SIZE} --output-dir {TOKENIZER_DIR}

# 3. Shard binary dataset
!python scripts/prepare_dataset.py --tokenizer-dir {TOKENIZER_DIR} --input-dir data/raw_code --output-dir data

## Step 8: Pretraining Run with Resumable Checkpoints

In [ ]:
#@title Step 8: Launch Pretraining { display-mode: "form" }
PRETRAIN_STEPS = 50 #@param {type:"integer"}
BATCH_SIZE = 2 #@param {type:"integer"}
RESUME_CHECKPOINT = "" #@param {type:"string"}

cmd = f"python scripts/train.py --config configs/{PROFILE}.yaml --train-bin data/train.bin --val-bin data/val.bin --max-steps {PRETRAIN_STEPS} --batch-size {BATCH_SIZE}"
if RESUME_CHECKPOINT.strip():
    cmd += f" --resume-from {RESUME_CHECKPOINT.strip()}"

print(f"Executing: {cmd}")
!{cmd}

## Step 9: Supervised Fine-Tuning (SFT with Prompt Loss Masking)

In [ ]:
#@title Step 9: Instruction Fine-Tuning { display-mode: "form" }
import glob

available_ckpts = sorted(glob.glob("checkpoints/*.pt"))
base_ckpt = available_ckpts[-1] if available_ckpts else ""
INSTRUCT_STEPS = 20 #@param {type:"integer"}

if base_ckpt:
    print(f"Fine-tuning base checkpoint: {base_ckpt}...")
    !python scripts/train_instruct.py --base-checkpoint {base_ckpt} --tokenizer-dir {TOKENIZER_DIR} --data-file data/coding_instructions.jsonl --max-steps {INSTRUCT_STEPS} --output-dir checkpoints/instruct
else:
    print("No base checkpoint found. Complete Step 8 first.")

## Step 10: Sandboxed Evaluation & Perplexity Verification

In [ ]:
#@title Step 10: Run Evaluation Benchmark { display-mode: "form" }
eval_ckpt = "checkpoints/instruct/checkpoint_instruct_final.pt"
if not os.path.exists(eval_ckpt) and available_ckpts:
    eval_ckpt = available_ckpts[-1]

if os.path.exists(eval_ckpt):
    !python scripts/evaluate.py --checkpoint {eval_ckpt} --tokenizer-dir {TOKENIZER_DIR} --val-bin data/val.bin
else:
    print("No checkpoint available for evaluation.")

## Step 11: Interactive Generation & AURA CLI Demo

In [ ]:
#@title Step 11: Text Generation Demo { display-mode: "form" }
PROMPT = "def fibonacci(n: int) -> int:\n" #@param {type:"string"}
if os.path.exists(eval_ckpt):
    !python scripts/generate.py --checkpoint {eval_ckpt} --tokenizer-dir {TOKENIZER_DIR} --prompt "{PROMPT}" --max-new-tokens 64

## Step 12: Sync Checkpoints to Google Drive

In [ ]:
#@title Step 12: Synchronize Checkpoints to Google Drive { display-mode: "form" }
if ENABLE_DRIVE_SYNC and os.path.exists("/content/drive/MyDrive"):
    print(f"Syncing checkpoints to Google Drive: {DRIVE_DIR}...")
    !python scripts/sync_checkpoints.py --src-dir checkpoints --dest-dir "{DRIVE_DIR}"
else:
    print("Drive synchronization disabled or Drive not mounted.")